# 02 - Back geometry inspector

**Question this notebook answers:** what exactly is being measured when the
pipeline says "arch"?

## The measurement

Five dorsal points are read from withers to sacrum:

`withers -> thoracic -> thoracolumbar -> lumbar -> sacrum`

A reference chord is drawn between the first and last point. Each intermediate
point's perpendicular distance to that chord is measured, and the largest is
normalized by the chord length:

$$S_n = \frac{\max_i d(P_i,\ \overline{\text{withers–sacrum}})}{\lVert \text{withers} - \text{sacrum} \rVert}$$

Normalizing by chord length is what makes the value comparable across animals
photographed at different distances and body sizes.

## The one rule of this notebook

> $S_n$ is a **feature that describes** arching. It is not an automatic label.

Thresholding `sagitta > 0.07 -> arched` to build the dataset and then training a
model on sagitta would only recover the threshold. The model would learn nothing,
and the metrics would be a measurement of the ruler by itself. Labels come from
notebook 03, from a human looking at the picture.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)
print("project root:", PROJECT_ROOT)

## Run configuration

In [ ]:
PREVIEW_ONLY = True
MAX_SAMPLES = 12
SAVE_OUTPUTS = False

MANIFEST_CSV = PROJECT_ROOT / "data" / "manifest.csv"
TRIM = 0.20          # fraction cut from each end of the silhouette topline

In [ ]:
def guard_save(what: str) -> bool:
    """Refuse to write anything while the notebook is in preview mode."""
    if PREVIEW_ONLY or not SAVE_OUTPUTS:
        print(f"PREVIEW MODE - not writing {what}. Set PREVIEW_ONLY=False and SAVE_OUTPUTS=True to commit.")
        return False
    return True

## 1. Two ways to measure the back

| | Silhouette topline (`auto_*`) | Five dorsal keypoints (`kp_*`) |
|---|---|---|
| Needs annotation | no | yes, ~5 clicks per image |
| Anatomically anchored | **no** - a fixed percentage trim | yes - withers and sacrum are named points |
| Fails when | head is down, tail swings, mask bleeds | annotator is inconsistent |
| Role in this project | experimental auxiliary feature | primary geometry for Model A |

The trim is why the silhouette version stays auxiliary: cutting 20% off each end
of the mask is a guess at where the body starts, not a localization of the
withers. A grazing cow's neck sits inside that window and lifts the curve.

In [ ]:
from cowarch.io import as_bool, read_manifest, resolve_data_path

manifest = read_manifest(MANIFEST_CSV)
accepted = manifest.loc[as_bool(manifest["accepted"])].reset_index(drop=True)
has_mask = accepted["mask_path"].astype(str).str.strip().ne("")
print(f"{len(accepted)} accepted crops, {int(has_mask.sum())} with a segmentation mask")

auto_columns = [c for c in ["auto_sagitta", "auto_chord_rmse", "auto_circle_curvature_norm"] if c in accepted.columns]
accepted[auto_columns] = accepted[auto_columns].apply(pd.to_numeric, errors="coerce")
accepted[auto_columns].describe().T

## 2. Inspect one sample end to end

Panel 5 shows the silhouette topline with its chord and the peak deviation.
Panel 6 shows the same idea on anatomical points, once they exist.

What to look for in panel 5:
- Does the orange topline follow the **back**, or does it climb the neck/head?
- Does the mask edge wobble where the coat meets a dark background?
- Is the peak deviation over the thoracolumbar region, or somewhere anatomically meaningless?

In [ ]:
import cv2

from cowarch.geometry import decode_keypoints
from cowarch.prepare import FrameOutcome
from cowarch.viz import inspect_frame, panel_keypoint_geometry, panel_topline

def load_outcome(row) -> FrameOutcome:
    """Rebuild an inspectable outcome from what the batch run wrote to disk."""
    crop = cv2.imread(str(resolve_data_path(str(row["crop_path"]), MANIFEST_CSV)))
    mask = None
    if str(row.get("mask_path", "")).strip():
        raw = cv2.imread(str(resolve_data_path(str(row["mask_path"]), MANIFEST_CSV)), cv2.IMREAD_GRAYSCALE)
        mask = None if raw is None else raw > 127
    return FrameOutcome(record=row.to_dict(), frame=crop, crop=crop, mask=mask,
                        detections=[], box=None)

def inspect_sample(row):
    outcome = load_outcome(row)
    points = decode_keypoints(row.get("keypoints_json", ""))
    fig = inspect_frame(outcome, keypoints=points, trim=TRIM)
    plt.show()

for _, row in accepted.loc[has_mask].head(3).iterrows():
    inspect_sample(row)

## 3. Where the silhouette measurement breaks

Sort by `auto_sagitta` and look at both ends. The top of the list is the honest
test of the metric: if the highest values are grazing cows and mask artefacts
rather than arched backs, the feature is measuring the wrong thing.

**This is diagnosis, not annotation.** Do not label from this ordering.

In [ ]:
if "auto_sagitta" in accepted.columns and accepted["auto_sagitta"].notna().any():
    ranked = accepted.loc[has_mask].dropna(subset=["auto_sagitta"]).sort_values("auto_sagitta")
    n = min(3, len(ranked))
    print("=== LOWEST auto_sagitta (silhouette says flat) ===")
    for _, row in ranked.head(n).iterrows():
        print(f"  {row['sample_id']}: {row['auto_sagitta']:.4f}")
        inspect_sample(row)
    print("=== HIGHEST auto_sagitta (silhouette says arched) ===")
    for _, row in ranked.tail(n).iloc[::-1].iterrows():
        print(f"  {row['sample_id']}: {row['auto_sagitta']:.4f}")
        inspect_sample(row)
else:
    print("no auto_sagitta values - the detector produced no masks")

### Confounders to name in the report

A raised topline is not always a painful back. Keep these in mind while looking:

- **Head down** (grazing, drinking): the dorsal line changes measurably, so these
  frames get labeled `uncertain` or `invalid` rather than `arched`.
- **Oblique or turning**: the silhouette is a projection of a rotated body.
- **Very thin animals**: spinal processes stand out and sharpen the outline.
- **Mask bleed**: dark coat against a dark background, or a rail crossing the back.

These are also the slices to break error analysis down by in notebook 04.

In [ ]:
distribution = accepted.loc[has_mask, "auto_sagitta"].dropna() if "auto_sagitta" in accepted.columns else pd.Series(dtype=float)
if len(distribution):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    axes[0].hist(distribution, bins=30, color="#2e86ab", edgecolor="white")
    axes[0].set_xlabel("auto_sagitta"); axes[0].set_ylabel("crops")
    axes[0].set_title("silhouette sagitta distribution")
    by_source = accepted.loc[has_mask].dropna(subset=["auto_sagitta"])
    by_source.boxplot(column="auto_sagitta", by="source_id", ax=axes[1], rot=90)
    axes[1].set_title("per source"); axes[1].set_xlabel("")
    fig.suptitle("")
    plt.tight_layout(); plt.show()
    print(
        "A large shift between sources is a warning: the model can separate "
        "sources instead of postures. This is why the split is by source group."
    )

## 4. Sanity-check the geometry code on synthetic curves

Before trusting the number on real animals, confirm it behaves on shapes whose
answer is known: a flat back must score ~0, and a deeper arch must score higher.

In [ ]:
from cowarch.geometry import keypoint_features

def synthetic_back(arch_px: float, length_px: float = 300.0) -> np.ndarray:
    t = np.linspace(0.0, 1.0, 5)
    return np.stack([t * length_px, 100.0 - arch_px * np.sin(np.pi * t)], axis=1)

rows = []
for arch in [0, 5, 10, 20, 30, 45]:
    features = keypoint_features(synthetic_back(arch))
    rows.append({"arch_px": arch, **{k: round(v, 4) for k, v in features.items()}})
check = pd.DataFrame(rows)
display(check)

monotone = check["kp_sagitta_norm"].is_monotonic_increasing
print(f"sagitta increases with arch depth: {monotone}")
assert monotone, "geometry is not responding monotonically to arch depth"

fig, ax = plt.subplots(figsize=(6, 3))
for arch in [0, 15, 35]:
    p = synthetic_back(arch)
    ax.plot(p[:, 0], p[:, 1], marker="o", label=f"arch={arch}px  S_n={keypoint_features(p)['kp_sagitta_norm']:.3f}")
ax.invert_yaxis(); ax.legend(fontsize=8); ax.set_title("synthetic dorsal lines")
plt.tight_layout(); plt.show()

## 5. Before you go on

You should now be able to answer, with a picture rather than a claim:

1. Does the mask cover the back correctly on this footage?
2. Does the topline follow the spine, or the neck?
3. Do the extreme sagitta values correspond to anything real?
4. Which confounders appear in *your* data?

Next: **03_labeling.ipynb**, where labels are actually created. The split must be
locked first, because splitting after labeling invites the choice to drift:

```bash
python scripts/03_split.py --manifest data/manifest.csv --group-column video_id
```